# Automatic Event Detection — Evaluation

전체 7경기에 대한 Open play / Set piece 탐지 성능 평가

| 항목 | 설정 |
|------|------|
| Open play 윈도우 | ±10s + 동일 player |
| Set piece 윈도우 | ±10s |
| 매칭 방식 | GT 기준 greedy 1:1 (Δt 최솟값) |

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
import pandas as pd

# pipeline 모듈 reload
import tools.config as cfg
import autoevent.setpiece_trigger as trg
import autoevent.set as st
import autoevent.pipeline as pl
import autoevent.open as op

importlib.reload(cfg)
importlib.reload(trg)
importlib.reload(st)
importlib.reload(pl)
importlib.reload(op)

from autoevent.pipeline import run_pipeline
from autoevent.open import OpenPlayEventDetector
from tools.evaluate import (
    eval_all_matches, eval_match, micro_summary,
    prepare_gt, prepare_pred, prepare_gt_goals, prepare_pred_goals,
    evaluate, evaluate_paper, evaluate_goals,
    OPEN_LABELS, SP_LABELS, PAPER_WINDOW, SP_WINDOW,
)
from tools.sportec_data import SportecData

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

print(f"R_PZ={cfg.R_PZ}  THROW_IN_BALL_Z_MIN={cfg.THROW_IN_BALL_Z_MIN}  SP_LOSS_MAP_WINDOW={cfg.SP_LOSS_MAP_WINDOW}")

R_PZ=2.5  THROW_IN_BALL_Z_MIN=0.5  SP_LOSS_MAP_WINDOW=5


## 1. 전체 경기 일괄 평가

In [ ]:
ALL_MATCH_IDS = ["J03WMX", "J03WN1", "J03WOH", "J03WOY", "J03WPY", "J03WQQ", "J03WR9"]

sum_open, sum_sp, sum_goal = eval_all_matches(
    ALL_MATCH_IDS,
    run_pipeline_fn=run_pipeline,
    open_play_detector_cls=OpenPlayEventDetector,
    open_window=PAPER_WINDOW,
    sp_window=SP_WINDOW,
    use_player=True,
)

Processing J03WMX... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WN1... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WOH... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WOY... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WPY... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WQQ... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WR9... Loading the tracking data...
Transforming the tracking data coordinates...
done


In [3]:
print(f"=== Open play 전경기 집계 (±{PAPER_WINDOW}s + player) ===")
print(sum_open.to_string())
m = micro_summary(sum_open)
print(f"\nMicro  TP={m['TP']} FP={m['FP']} FN={m['FN']}  P={m['Precision']:.3f} R={m['Recall']:.3f} F1={m['F1']:.3f}")

=== Open play 전경기 집계 (±10.0s + player) ===
                GT  Pred    TP    FP    FN  Precision  Recall     F1
label                                                               
cross          229   149   100    49   129      0.671   0.437  0.529
interception   146  1827   110  1717    36      0.060   0.753  0.111
pass          5833  4511  3912   599  1921      0.867   0.671  0.757
shot           171    76    38    38   133      0.500   0.222  0.307

Micro  TP=4160 FP=2403 FN=2219  P=0.634 R=0.652 F1=0.643


In [4]:
print(f"=== Set piece 전경기 집계 (±{SP_WINDOW}s) ===")
print(sum_sp.to_string())
m = micro_summary(sum_sp)
print(f"\nMicro  TP={m['TP']} FP={m['FP']} FN={m['FN']}  P={m['Precision']:.3f} R={m['Recall']:.3f} F1={m['F1']:.3f}")

=== Set piece 전경기 집계 (±10.0s) ===
               GT  Pred   TP  FP  FN  Precision  Recall     F1
label                                                         
corner_kick    57    46   46   0  11      1.000   0.807  0.893
free_kick     172   175  147  28  25      0.840   0.855  0.847
goal_kick     119   114  109   5  10      0.956   0.916  0.936
kickoff        32    21   21   0  11      1.000   0.656  0.792
penalty_kick    3     3    3   0   0      1.000   1.000  1.000
throw_in      308   293  275  18  33      0.939   0.893  0.915

Micro  TP=601 FP=51 FN=90  P=0.922 R=0.870 F1=0.895


In [ ]:
print(f"=== 골 검출 전경기 집계 (±{SP_WINDOW}s) ===")
print(sum_goal.to_string())
m = micro_summary(sum_goal)
print(f"\nMicro  TP={m['TP']} FP={m['FP']} FN={m['FN']}  P={m['Precision']:.3f} R={m['Recall']:.3f} F1={m['F1']:.3f}")

In [9]:
# R_PZ=1.0 으로 재실행
cfg.R_PZ = 1.0
importlib.reload(cfg)

sum_open_rpz1, sum_sp_rpz1 = eval_all_matches(
    ALL_MATCH_IDS,
    run_pipeline_fn=run_pipeline,
    open_play_detector_cls=OpenPlayEventDetector,
    open_window=PAPER_WINDOW,
    sp_window=SP_WINDOW,
    use_player=True,
)

print(f"=== Open play 전경기 집계 (R_PZ=1.0, ±{PAPER_WINDOW}s + player) ===")
print(sum_open_rpz1.to_string())
m = micro_summary(sum_open_rpz1)
print(f"\nMicro  TP={m['TP']} FP={m['FP']} FN={m['FN']}  P={m['Precision']:.3f} R={m['Recall']:.3f} F1={m['F1']:.3f}")

print(f"\n=== Set piece 전경기 집계 (R_PZ=1.0, ±{SP_WINDOW}s) ===")
print(sum_sp_rpz1.to_string())
m = micro_summary(sum_sp_rpz1)
print(f"\nMicro  TP={m['TP']} FP={m['FP']} FN={m['FN']}  P={m['Precision']:.3f} R={m['Recall']:.3f} F1={m['F1']:.3f}")

# 원래 값 복원
cfg.R_PZ = 2.5
importlib.reload(cfg)

Processing J03WMX... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WN1... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WOH... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WOY... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WPY... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WQQ... Loading the tracking data...
Transforming the tracking data coordinates...
done
Processing J03WR9... Loading the tracking data...
Transforming the tracking data coordinates...
done
=== Open play 전경기 집계 (R_PZ=1.0, ±10.0s + player) ===
                GT  Pred    TP    FP    FN  Precision  Recall     F1
label                                                               
cross          229   149   100    49   129      0.671   0.437  0.529
interception   146  1827   110  1

<module 'tools.config' from '/Users/kimseungwoo/Desktop/시립대/Automatic Event Detection/tools/config.py'>

In [10]:

print(f"=== Open play 전경기 집계 (R_PZ=1.0, ±{PAPER_WINDOW}s + player) ===")
print(sum_open_rpz1.to_string())
m = micro_summary(sum_open_rpz1)
print(f"\nMicro  TP={m['TP']} FP={m['FP']} FN={m['FN']}  P={m['Precision']:.3f} R={m['Recall']:.3f} F1={m['F1']:.3f}")

print(f"\n=== Set piece 전경기 집계 (R_PZ=1.0, ±{SP_WINDOW}s) ===")
print(sum_sp_rpz1.to_string())
m = micro_summary(sum_sp_rpz1)
print(f"\nMicro  TP={m['TP']} FP={m['FP']} FN={m['FN']}  P={m['Precision']:.3f} R={m['Recall']:.3f} F1={m['F1']:.3f}")


=== Open play 전경기 집계 (R_PZ=1.0, ±10.0s + player) ===
                GT  Pred    TP    FP    FN  Precision  Recall     F1
label                                                               
cross          229   149   100    49   129      0.671   0.437  0.529
interception   146  1827   110  1717    36      0.060   0.753  0.111
pass          5833  4511  3912   599  1921      0.867   0.671  0.757
shot           171    76    38    38   133      0.500   0.222  0.307

Micro  TP=4160 FP=2403 FN=2219  P=0.634 R=0.652 F1=0.643

=== Set piece 전경기 집계 (R_PZ=1.0, ±10.0s) ===
               GT  Pred   TP  FP  FN  Precision  Recall     F1
label                                                         
corner_kick    57    46   46   0  11      1.000   0.807  0.893
free_kick     172   175  147  28  25      0.840   0.855  0.847
goal_kick     119   114  109   5  10      0.956   0.916  0.936
kickoff        32    21   21   0  11      1.000   0.656  0.792
penalty_kick    3     3    3   0   0      1.000   1.

## 2. 단일 경기 상세 평가

In [15]:
MATCH_ID = "J03WMX"  # 분석할 경기 ID 변경 가능

m_single = SportecData(MATCH_ID)
r_single = run_pipeline(m_single.tracking)
t_single = OpenPlayEventDetector(r_single).run()

open_rows, sp_rows, goal_row_single = eval_match(
    m_single, r_single, t_single,
    open_window=PAPER_WINDOW,
    sp_window=SP_WINDOW,
)

open_df = pd.DataFrame(open_rows).set_index("label")
sp_df   = pd.DataFrame(sp_rows).set_index("label")

print(f"=== {MATCH_ID} — Open play (±{PAPER_WINDOW}s + player) ===")
print(open_df.to_string())
m = micro_summary(open_df)
print(f"\nMicro  P={m['Precision']:.3f} R={m['Recall']:.3f} F1={m['F1']:.3f}")

print(f"\n=== {MATCH_ID} — Set piece (±{SP_WINDOW}s) ===")
print(sp_df.to_string())
m = micro_summary(sp_df)
print(f"\nMicro  P={m['Precision']:.3f} R={m['Recall']:.3f} F1={m['F1']:.3f}")


Loading the tracking data...
Transforming the tracking data coordinates...
=== J03WMX — Open play (±10.0s + player) ===
               GT  Pred   TP   FP   FN  Precision  Recall     F1
label                                                           
pass          951   751  636  115  315      0.847   0.669  0.747
cross          33    22   14    8   19      0.636   0.424  0.509
shot           21     9    3    6   18      0.333   0.143  0.200
interception   18   299   16  283    2      0.054   0.889  0.101

Micro  P=0.619 R=0.654 F1=0.636

=== J03WMX — Set piece (±10.0s) ===
              GT  Pred  TP  FP  FN  Precision  Recall     F1
label                                                       
throw_in      52    54  52   2   0      0.963     1.0  0.981
goal_kick     14    15  14   1   0      0.933     1.0  0.966
corner_kick   10    10  10   0   0      1.000     1.0  1.000
free_kick     20    18  16   2   4      0.889     0.8  0.842
kickoff        5     5   5   0   0      1.000     1.0 

## 3. FN / FP 상세 분석 (단일 경기)

In [6]:
# Set piece 타입별 FN / FP 타임스탬프 출력
from tools.evaluate import SP_MAP_GT
from tools.match_data import MatchData

gt_eval, gt_sp = prepare_gt(m_single)
pred_full, pred_sp = prepare_pred(r_single, t_single)

for label in SP_LABELS:
    gt_sub   = gt_sp[gt_sp["label"] == label][["period_id","timestamp","object_id"]].reset_index(drop=True)
    pred_sub = pred_sp[pred_sp["label"] == label][["period_id","timestamp"]].reset_index(drop=True)

    matched_gt, matched_pr = set(), set()
    for gi, gr in gt_sub.iterrows():
        for pi, pr_ in pred_sub.iterrows():
            if gr["period_id"] == pr_["period_id"] and abs(gr["timestamp"] - pr_["timestamp"]) <= SP_WINDOW:
                if gi not in matched_gt and pi not in matched_pr:
                    matched_gt.add(gi); matched_pr.add(pi)

    tp = len(matched_gt)
    fp = len(pred_sub) - len(matched_pr)
    fn = len(gt_sub) - tp
    p  = tp/(tp+fp) if (tp+fp) > 0 else 0
    r  = tp/(tp+fn) if (tp+fn) > 0 else 0
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0
    print(f"=== {label}  GT={len(gt_sub)} Pred={len(pred_sub)} TP={tp} FP={fp} FN={fn}  P={p:.3f} R={r:.3f} F1={f1:.3f}")

    if fn > 0:
        print("  [FN — 탐지 못한 GT]")
        for i, g in gt_sub.iterrows():
            if i not in matched_gt:
                same = pred_sub[pred_sub["period_id"] == g["period_id"]]
                dt = f"{(same['timestamp'] - g['timestamp']).abs().min():.1f}s" if len(same) else "no pred"
                print(f"    p={g['period_id']} t={g['timestamp']:.1f}s player={g['object_id']}  (closest Δt={dt})")

    if fp > 0:
        print("  [FP — 잘못 탐지한 Pred]")
        for i, p_ in pred_sub.iterrows():
            if i not in matched_pr:
                same = gt_sub[gt_sub["period_id"] == p_["period_id"]]
                dt = f"{(same['timestamp'] - p_['timestamp']).abs().min():.1f}s" if len(same) else "no GT"
                print(f"    p={p_['period_id']} t={p_['timestamp']:.1f}s  (closest GT Δt={dt})")
    print()

=== throw_in  GT=52 Pred=54 TP=52 FP=2 FN=0  P=0.963 R=1.000 F1=0.981
  [FP — 잘못 탐지한 Pred]
    p=2.0 t=579.9s  (closest GT Δt=46.3s)
    p=2.0 t=2541.4s  (closest GT Δt=87.5s)

=== goal_kick  GT=14 Pred=15 TP=14 FP=1 FN=0  P=0.933 R=1.000 F1=0.966
  [FP — 잘못 탐지한 Pred]
    p=1.0 t=759.2s  (closest GT Δt=204.9s)

=== corner_kick  GT=10 Pred=10 TP=10 FP=0 FN=0  P=1.000 R=1.000 F1=1.000

=== free_kick  GT=20 Pred=18 TP=16 FP=2 FN=4  P=0.889 R=0.800 F1=0.842
  [FN — 탐지 못한 GT]
    p=1 t=760.7s player=away_27  (closest Δt=478.2s)
    p=2 t=579.8s player=home_2  (closest Δt=231.4s)
    p=2 t=2397.1s player=away_5  (closest Δt=121.4s)
    p=2 t=2543.5s player=away_22  (closest Δt=25.0s)
  [FP — 잘못 탐지한 Pred]
    p=1.0 t=249.1s  (closest GT Δt=89.4s)
    p=2.0 t=1667.9s  (closest GT Δt=562.3s)

=== kickoff  GT=5 Pred=5 TP=5 FP=0 FN=0  P=1.000 R=1.000 F1=1.000

=== penalty_kick  GT=1 Pred=1 TP=1 FP=0 FN=0  P=1.000 R=1.000 F1=1.000



## 4. 미할당 구간 진단 (단일 경기)

In [7]:
from autoevent.poss import PossessionDetector
from autoevent.set import SetPieceDetector
from collections import Counter

poss_chk = PossessionDetector(m_single.tracking).run()
d_chk = SetPieceDetector(poss_chk)
d_chk.add_dead_ball_intervals()
d_chk.add_first_in_frames()
d_chk.add_kickoff_labels()
d_chk.add_penalty_labels()
d_chk.add_corner_labels()
d_chk.add_throw_in_labels()
d_chk.add_goal_kick_labels()
d_chk.add_free_kick_labels()
d_chk._add_fallback_labels()

no_sp = []
for iv in d_chk.intervals:
    fi = iv["first_in_idx"]
    sp = d_chk.tracking.at[fi, "set_piece_type"]
    if pd.isna(sp):
        db_ev = (d_chk.tracking.loc[iv["start_idx"]:iv["end_idx"], "deadball_event"]
                 .dropna().unique().tolist()) if iv.get("start_idx") is not None else []
        ts  = d_chk.tracking.at[fi, "timestamp"]
        pid = d_chk.tracking.at[fi, "period_id"]
        no_sp.append({"deadball_id": iv["deadball_id"], "period": pid,
                      "first_in_ts": round(float(ts), 2), "deadball_events": db_ev})

print(f"set_piece_type 미할당 구간: {len(no_sp)}개 / 전체 {len(d_chk.intervals)}개")
if no_sp:
    ev_cnt = Counter(ev for row in no_sp for ev in row["deadball_events"])
    print("\n미할당 구간 deadball_event 유형:")
    for ev, cnt in ev_cnt.most_common():
        print(f"  {ev!r}: {cnt}건")
    print("\n상세 목록:")
    for row in no_sp:
        print(f"  db={row['deadball_id']:3d}  p={row['period']}  t={row['first_in_ts']:7.2f}s  {row['deadball_events']}")
else:
    print("모든 구간에 set_piece_type 할당 완료!")

set_piece_type 미할당 구간: 0개 / 전체 103개
모든 구간에 set_piece_type 할당 완료!


## 5. 검출 이벤트 종합 (단일 경기)

In [8]:
# Section 2에서 m_single / r_single / t_single 이 로드되어 있어야 함

gt_eval2, gt_sp2 = prepare_gt(m_single)
pred_full2, pred_sp2 = prepare_pred(r_single, t_single)

print(f"=== {MATCH_ID} 검출 이벤트 value_counts ===\n")

print("--- [Open play] t_single event_name (raw) ---")
print(t_single["event_name"].value_counts(dropna=False).to_string())

print("\n--- [Open play] pred_full2 label (평가용) ---")
print(pred_full2["label"].value_counts().to_string() if len(pred_full2) else "(없음)")

print("\n--- [Set piece] pred_sp2 label ---")
print(pred_sp2["label"].value_counts().to_string() if len(pred_sp2) else "(없음)")

=== J03WMX 검출 이벤트 value_counts ===

--- [Open play] t_single event_name (raw) ---
event_name
<NA>               144058
reception             828
pass                  751
interception          299
cross                  22
shot_off_target         7
shot_on_target          2

--- [Open play] pred_full2 label (평가용) ---
label
pass            751
interception    299
cross            22
shot              9

--- [Set piece] pred_sp2 label ---
label
throw_in        54
free_kick       18
goal_kick       15
corner_kick     10
kickoff          5
penalty_kick     1


## 6. 골 검출 평가 (단일 경기)

In [16]:
# Section 2에서 goal_row_single 이 로드되어 있어야 함

print(f"=== {MATCH_ID} — 골 검출 (±{SP_WINDOW}s) ===")
print(f"GT 골: {goal_row_single['GT']}개  Pred 골: {goal_row_single['Pred']}개")
print(f"TP={goal_row_single['TP']}  FP={goal_row_single['FP']}  FN={goal_row_single['FN']}")
print(f"Precision={goal_row_single['Precision']:.3f}  Recall={goal_row_single['Recall']:.3f}  F1={goal_row_single['F1']:.3f}")


=== J03WMX — 골 검출 (±10.0s) ===
GT 골: 3개  Pred 골: 3개
TP=3  FP=0  FN=0
Precision=1.000  Recall=1.000  F1=1.000
